In [1]:
import mph
import numpy as np
from scipy.optimize import minimize, differential_evolution
import time

In [2]:
class SurfaceTrapOptimizer:
    def __init__(self, model_path):
        print("Starting COMSOL client...")
        self.client = mph.start(cores=1)
        self.model = self.client.load(model_path)
        self.optimization_history = []
        self.failed_simulations = 0
        self.max_failures = 5
        self.cache = {}
        
        # Default parameters from Editable_Param_SurfaceTrap.txt
        self.default_params = {
            'center_width': 0.1143, 'rf_width': 0.1929, 'rf_spacing': 0.1929,
            'V_rf': 328.5714, 'V_dc': 97.1429, 'endcap_rad': 3.0000,
            'endcap_thick': 0.2000, 'endcap_offset': 0.5000, 
            'V_endcap': 0.1000, 'f': 30.0000, 'length': 1.2143
        }
        
        # Baseline metrics from SurfaceTrap_Metrics.txt
        self.baseline_metrics = {
            'depth_eV': 24.358714381550527,
            'offset_mm': 0.4230873423493664,
            'P_est_mW': 0.002188169509449676,
            'trap_x': 0,
            'trap_y': -3.946475e-4,
            'trap_z': 1.525e-4
        }
    
    def comprehensive_sweep_analysis(self):
        """
        Comprehensive sweep to understand parameter relationships and effects
        """
        print("Starting Parameter Sweep...")
        
        base_params = list(self.default_params.values())
        param_names = list(self.default_params.keys())
        
        sweep_results = {}
        relationship_insights = []
        
        # Wider sweep ranges for surface trap exploration
        sweep_config = {
            'center_width': {'points': 8, 'range': (0.05, 0.5)},
            'rf_width': {'points': 8, 'range': (0.05, 0.3)},
            'rf_spacing': {'points': 8, 'range': (0.05, 0.3)},
            'V_rf': {'points': 8, 'range': (100, 500)},
            'V_dc': {'points': 8, 'range': (20, 200)},
            'endcap_rad': {'points': 6, 'range': (3, 10)},
            'endcap_thick': {'points': 6, 'range': (0.2, 1.0)},
            'endcap_offset': {'points': 8, 'range': (0.5, 3.0)},
            'V_endcap': {'points': 8, 'range': (0.1, 10)},
            'f': {'points': 8, 'range': (5, 30)},
            'length': {'points': 8, 'range': (0.5, 3.0)}
        }
        
        # Sweep each parameter and analyze relationships
        for i, param_name in enumerate(param_names):
            print(f"\nSweeping {param_name}...")
            config = sweep_config[param_name]
            
            if isinstance(config['range'][0], (int, float)):
                values = np.linspace(config['range'][0], config['range'][1], config['points'])
            else:
                base_value = base_params[i]
                low_ratio, high_ratio = config['range']
                values = np.linspace(base_value * low_ratio, base_value * high_ratio, config['points'])
            
            param_results = []
            
            for value in values:
                test_params = base_params.copy()
                test_params[i] = value
                
                metrics = self.safe_simulation(test_params)
                objective = self.calculate_multi_objective(metrics)
                
                result = {
                    'value': value,
                    'objective': objective,
                    'depth_eV': metrics['depth_eV'],
                    'offset_mm': metrics['offset_mm'],
                    'P_est_mW': metrics['P_est_mW'],
                    'trap_y': metrics['trap_y'],
                    'trap_z': metrics['trap_z']
                }
                param_results.append(result)
            
            # Analyze relationships for this parameter
            insights = self.analyze_parameter_relationships(param_name, param_results)
            relationship_insights.extend(insights)
            
            sweep_results[param_name] = {
                'results': param_results,
                'best_value': min(param_results, key=lambda x: x['objective'])['value'],
                'best_objective': min(param_results, key=lambda x: x['objective'])['objective']
            }
            
            best_result = min(param_results, key=lambda x: x['objective'])
            print(f"  Best {param_name}: {best_result['value']:.4f}")
            print(f"  Depth: {best_result['depth_eV']:.2f}eV, Offset: {best_result['offset_mm']:.4f}mm")
        
        # Build smart bounds based on sweep insights
        smart_bounds = self.build_smart_bounds(sweep_results)
        
        return sweep_results, relationship_insights, smart_bounds
    
    def analyze_parameter_relationships(self, param_name, results):
        """
        Analyze relationships between parameters and performance metrics
        """
        insights = []
        
        depths = [r['depth_eV'] for r in results]
        offsets = [r['offset_mm'] for r in results]
        powers = [r['P_est_mW'] for r in results]
        values = [r['value'] for r in results]
        
        # Calculate correlations
        depth_correlation = np.corrcoef(values, depths)[0,1] if len(values) > 1 else 0
        offset_correlation = np.corrcoef(values, offsets)[0,1] if len(values) > 1 else 0
        power_correlation = np.corrcoef(values, powers)[0,1] if len(values) > 1 else 0
        
        # Generate insights based on correlations
        if abs(depth_correlation) > 0.7:
            trend = "increases" if depth_correlation > 0 else "decreases"
            insights.append(f" {param_name} strongly {trend} trap depth")
        
        if abs(offset_correlation) > 0.7:
            trend = "increases" if offset_correlation > 0 else "decreases" 
            insights.append(f"{param_name} strongly {trend} trap offset")
        
        if abs(power_correlation) > 0.7:
            trend = "increases" if power_correlation > 0 else "decreases"
            insights.append(f" {param_name} strongly {trend} RF power")
        
        # Find optimal ranges
        good_depth_threshold = max(depths) * 0.8  # Top 20% of depths
        good_offset_threshold = min(offsets) * 1.5  # Within 50% of best offset
        
        good_depth_values = [v for v, d in zip(values, depths) if d >= good_depth_threshold]
        good_offset_values = [v for v, o in zip(values, offsets) if o <= good_offset_threshold]
        
        if good_depth_values:
            insights.append(f"Good depth range for {param_name}: {min(good_depth_values):.3f} to {max(good_depth_values):.3f}")
        if good_offset_values:
            insights.append(f"Good offset range for {param_name}: {min(good_offset_values):.3f} to {max(good_offset_values):.3f}")
        
        return insights
    
    def build_smart_bounds(self, sweep_results):
        """
        Build intelligent bounds based on sweep analysis
        """
        param_names = list(sweep_results.keys())
        smart_bounds = []
        
        # Default wider bounds for surface trap exploration
        wide_bounds = [
            (0.05, 0.5),    # center_width
            (0.05, 0.4),    # rf_width  
            (0.05, 0.4),    # rf_spacing
            (80, 400),      # V_rf
            (30, 150),      # V_dc
            (3, 12),        # endcap_rad
            (0.2, 1.2),     # endcap_thick
            (0.3, 2.5),     # endcap_offset
            (0.5, 15),      # V_endcap
            (5, 25),        # f
            (0.8, 2.5)      # length
        ]
        
        # Refine bounds based on sweep results
        for i, param_name in enumerate(param_names):
            best_value = sweep_results[param_name]['best_value']
            results = sweep_results[param_name]['results']
            
            # Find the range where objective is within 20% of best
            best_objective = sweep_results[param_name]['best_objective']
            threshold = best_objective * 1.2
            
            good_values = [r['value'] for r in results if r['objective'] <= threshold]
            
            if good_values:
                # Use the good range but expand slightly for exploration
                range_min = max(wide_bounds[i][0], min(good_values) * 0.8)
                range_max = min(wide_bounds[i][1], max(good_values) * 1.2)
                
                # Ensure minimum range width
                min_range_width = (wide_bounds[i][1] - wide_bounds[i][0]) * 0.3
                if (range_max - range_min) < min_range_width:
                    center = (range_min + range_max) / 2
                    range_min = center - min_range_width / 2
                    range_max = center + min_range_width / 2
                
                smart_bounds.append((range_min, range_max))
            else:
                smart_bounds.append(wide_bounds[i])
        
        return smart_bounds
    
    def calculate_multi_objective(self, metrics):
        """
        Multi-objective function: MAXIMIZE depth_eV, MINIMIZE offset_mm and P_est_mW
        """
        depth = metrics['depth_eV']
        offset = metrics['offset_mm']
        power = metrics['P_est_mW']
        
        # Normalize by baseline values
        depth_norm = depth / self.baseline_metrics['depth_eV']  # We want this HIGH
        offset_norm = offset / self.baseline_metrics['offset_mm']  # We want this LOW
        power_norm = power / self.baseline_metrics['P_est_mW']  # We want this LOW
        
        # Combined objective: minimize this function
        # We maximize depth by using (1/depth_norm) in minimization
        # We minimize offset and power by using them directly
        objective = (
            0.5 * (1.0 / max(depth_norm, 0.1)) +  # Maximize depth (avoid division by zero)
            0.3 * offset_norm +                   # Minimize offset
            0.2 * np.log1p(power_norm)            # Minimize power (log scale for large variations)
        )
        
        # Penalties for undesirable conditions
        penalty = 0.0
        
        # Heavy penalty if trap depth is too small
        if depth < 50.0:
            penalty += 20.0 * (1.0 - depth)
            
        # Penalize large offsets
        if offset > 1.0:
            penalty += 10.0 * (offset - 1.0)
            
        # Penalize if trap is not near surface (trap_z should be small for surface trap)
        if abs(metrics['trap_z']) > 0.1:
            penalty += 5.0 * abs(metrics['trap_z'])
            
        return objective + penalty
    
    def smart_optimization(self, bounds, max_iterations=20):
        """
        Smart optimization using insights from sweep analysis
        """
        print("\nStarting Optimization...")
        
        # Start from default parameters
        initial_guess = list(self.default_params.values())
        
        initial_guess = list(self.default_params.values())
    
        # Single fast local optimization
        result = minimize(
            self.objective_function,
            initial_guess,
            bounds=bounds,
            method='Nelder-Mead',
            options={
                'maxiter': 8,
                'disp': True,
                'xatol': 0.2,
                'fatol': 0.2,
                'adaptive': True
            }
        )
        
        return result
    
    def objective_function(self, params):
        """
        Objective function wrapper with caching
        """
        params_key = tuple(params)
        if params_key in self.cache:
            return self.cache[params_key]
        
        metrics = self.safe_simulation(params)
        objective = self.calculate_multi_objective(metrics)
        
        self.cache[params_key] = objective
        
        self.optimization_history.append({
            'params': params.copy(),
            'metrics': metrics.copy(),
            'objective': objective
        })
        
        print(f"  Depth: {metrics['depth_eV']:.2f}eV, "
              f"Offset: {metrics['offset_mm']:.4f}mm, "
              f"Power: {metrics['P_est_mW']:.6f}mW")
        
        return objective
    
    def safe_simulation(self, params):
        """Run simulation with error handling"""
        try:
            self.update_parameters(params)
            self.model.solve()
            metrics = self.extract_trap_metrics()
            
            if metrics is None:
                raise ValueError("Failed to extract metrics")
            
            self.failed_simulations = 0
            return metrics
            
        except Exception as e:
            self.failed_simulations += 1
            print(f"  Simulation failed: {e}")
            
            if self.failed_simulations >= self.max_failures:
                raise RuntimeError(f"Too many consecutive failures ({self.max_failures})")
            
            return self.get_conservative_metrics()
    
    def get_conservative_metrics(self):
        """Return conservative metrics for failed simulations"""
        return {
            'depth_eV': 0.1, 'offset_mm': 10.0, 'P_est_mW': 1000.0,
            'trap_x': 5.0, 'trap_y': 5.0, 'trap_z': 5.0
        }
    
    def update_parameters(self, params):
        """Update all surface trap parameters in COMSOL"""
        parameter_definitions = [
            ('center_width', params[0], 'mm'),
            ('rf_width', params[1], 'mm'),
            ('rf_spacing', params[2], 'mm'),
            ('V_rf', params[3], 'V'),
            ('V_dc', params[4], 'V'),
            ('endcap_rad', params[5], 'mm'),
            ('endcap_thick', params[6], 'mm'),
            ('endcap_offset', params[7], 'mm'),
            ('V_endcap', params[8], 'V'),
            ('f', params[9], 'MHz'),
            ('length', params[10], 'mm')
        ]
        
        for param_name, value, unit in parameter_definitions:
            self.model.parameter(param_name, f'{value}[{unit}]')
    
    def extract_trap_metrics(self):
        """Extract trap metrics from COMSOL"""
        try:
            return {
                'depth_eV': float(self.model.evaluate('depth_eV')),
                'offset_mm': float(self.model.evaluate('offset_mm')),
                'P_est_mW': float(self.model.evaluate('P_est_mW')),
                'trap_x': float(self.model.evaluate('trap_x')),
                'trap_y': float(self.model.evaluate('trap_y')), 
                'trap_z': float(self.model.evaluate('trap_z'))
            }
        except Exception as e:
            print(f"Error extracting metrics: {e}")
            return None
    
    def get_best_results(self):
        """Get the best optimization results"""
        if not self.optimization_history:
            return None, None
        best_entry = min(self.optimization_history, key=lambda x: x['objective'])
        return best_entry['params'], best_entry['metrics']
    
    def export_optimization_report(self, sweep_results, relationship_insights, filename='surface_trap_optimization_report.txt'):
        """Comprehensive optimization report"""
        params, metrics = self.get_best_results()
        if params is None:
            print("No results to export")
            return
        
        param_names = list(self.default_params.keys())
        
        with open(filename, 'w') as f:
            f.write("="*80 + "\n")
            f.write("SURFACE TRAP OPTIMIZATION REPORT\n")
            f.write("="*80 + "\n\n")
            
            f.write("OPTIMIZATION STRATEGY:\n")
            f.write("- Multi-objective: MAXIMIZE depth_eV, MINIMIZE offset_mm and P_est_mW\n")
            f.write("- Comprehensive parameter sweep analysis\n")
            f.write("- Relationship-based smart bounds\n")
            f.write("- Two-phase optimization (global + local)\n\n")
            
            f.write("KEY RELATIONSHIP INSIGHTS:\n")
            f.write("-" * 50 + "\n")
            for insight in relationship_insights:
                f.write(f"{insight}\n")
            f.write("\n")
            
            f.write("OPTIMIZED PARAMETERS:\n")
            f.write("-" * 50 + "\n")
            for name, value in zip(param_names, params):
                f.write(f"{name:15}: {value:10.6f}\n")
            
            f.write("\nPERFORMANCE COMPARISON:\n")
            f.write("-" * 50 + "\n")
            f.write("Metric           | Baseline     | Optimized   | Improvement\n")
            f.write("-" * 50 + "\n")
            
            depth_improve = ((metrics['depth_eV'] - self.baseline_metrics['depth_eV']) / self.baseline_metrics['depth_eV']) * 100
            offset_improve = ((self.baseline_metrics['offset_mm'] - metrics['offset_mm']) / self.baseline_metrics['offset_mm']) * 100
            power_improve = ((self.baseline_metrics['P_est_mW'] - metrics['P_est_mW']) / self.baseline_metrics['P_est_mW']) * 100
            
            f.write(f"depth_eV (eV)    | {self.baseline_metrics['depth_eV']:11.2f} | {metrics['depth_eV']:11.2f} | {depth_improve:+.1f}%\n")
            f.write(f"offset_mm (mm)   | {self.baseline_metrics['offset_mm']:11.4f} | {metrics['offset_mm']:11.4f} | {offset_improve:+.1f}%\n")
            f.write(f"P_est_mW (mW)    | {self.baseline_metrics['P_est_mW']:11.6f} | {metrics['P_est_mW']:11.6f} | {power_improve:+.1f}%\n")
            
            f.write(f"\nTrap Center Coordinates:\n")
            f.write(f"  trap_x: {metrics['trap_x']:.6f} mm\n")
            f.write(f"  trap_y: {metrics['trap_y']:.6f} mm\n") 
            f.write(f"  trap_z: {metrics['trap_z']:.6f} mm\n")
            
            f.write(f"\nSWEEP ANALYSIS SUMMARY (Best Parameters):\n")
            f.write("-" * 50 + "\n")
            for param_name in param_names:
                best_sweep = sweep_results[param_name]['best_value']
                f.write(f"{param_name:15}: {best_sweep:10.6f}\n")
        
        print(f"✅ Optimization report saved to {filename}")
        return params, metrics

def main():
    print("SURFACE TRAP COMPREHENSIVE OPTIMIZATION")
    print("=" * 60)
    
    start_time = time.time()
    optimizer = SurfaceTrapOptimizer('Surface_trap(v4).mph')
    
    try:
        # Step 1: Comprehensive sweep analysis
        print("STEP 1: Parameter Sweep & Relationship Analysis")
        print("=" * 60)
        sweep_results, relationship_insights, smart_bounds = optimizer.comprehensive_sweep_analysis()
        
        # Print key insights
        print("\nKEY RELATIONSHIP INSIGHTS:")
        print("-" * 50)
        for insight in relationship_insights[:10]:  # Show top 10 insights
            print(insight)
        
        # Step 2: Smart optimization
        print("\nSTEP 2: Smart Optimization with Relationship Insights")
        print("=" * 60)
        result = optimizer.smart_optimization(smart_bounds, max_iterations=20)
        
        # Step 3: Export comprehensive report
        print("\nSTEP 3: Generating Comprehensive Report")
        print("=" * 60)
        optimized_params, optimized_metrics = optimizer.export_optimization_report(
            sweep_results, relationship_insights
        )
        
        total_time = time.time() - start_time
        total_simulations = len(optimizer.optimization_history)
        
        print("\n" + "="*60)
        print("SURFACE TRAP OPTIMIZATION COMPLETED!")
        print("="*60)
        print(f"Total time: {total_time:.1f} seconds")
        print(f"Total simulations: {total_simulations}")
        print(f"Final objective: {result.fun:.4f}")
        print(f"Optimized depth_eV: {optimized_metrics['depth_eV']:.2f} eV")
        print(f"Optimized offset_mm: {optimized_metrics['offset_mm']:.4f} mm")
        print(f"Optimized power: {optimized_metrics['P_est_mW']:.6f} mW")
        print(f"Trap center: ({optimized_metrics['trap_x']:.3f}, {optimized_metrics['trap_y']:.3f}, {optimized_metrics['trap_z']:.3f}) mm")
        
    except Exception as e:
        print(f"\nOptimization interrupted: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        print("\nCleaning up...")
        optimizer.client.disconnect()

if __name__ == "__main__":
    main()

SURFACE TRAP COMPREHENSIVE OPTIMIZATION
Starting COMSOL client...
STEP 1: Parameter Sweep & Relationship Analysis
Starting Parameter Sweep...

Sweeping center_width...


/usr/lib64/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/lib64/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  Best center_width: 0.0500
  Depth: 28.90eV, Offset: 0.4231mm

Sweeping rf_width...
  Best rf_width: 0.3000
  Depth: 26.03eV, Offset: 0.4231mm

Sweeping rf_spacing...
  Best rf_spacing: 0.1571
  Depth: 24.50eV, Offset: 0.4231mm

Sweeping V_rf...
  Best V_rf: 500.0000
  Depth: 32.05eV, Offset: 0.4213mm

Sweeping V_dc...
  Best V_dc: 200.0000
  Depth: 35.12eV, Offset: 0.4231mm

Sweeping endcap_rad...
  Best endcap_rad: 3.0000
  Depth: 24.36eV, Offset: 0.4231mm

Sweeping endcap_thick...
  Best endcap_thick: 0.2000
  Depth: 24.36eV, Offset: 0.4231mm

Sweeping endcap_offset...
  Best endcap_offset: 0.5000
  Depth: 24.36eV, Offset: 0.4231mm

Sweeping V_endcap...
  Best V_endcap: 0.1000
  Depth: 24.36eV, Offset: 0.4231mm

Sweeping f...
  Best f: 30.0000
  Depth: 24.36eV, Offset: 0.4231mm

Sweeping length...
  Best length: 0.8571
  Depth: 27.60eV, Offset: 0.3176mm

KEY RELATIONSHIP INSIGHTS:
--------------------------------------------------
 center_width strongly decreases trap depth
Good de

/tmp/ipykernel_155601/2884008902.py:252: OptimizeWarning: Initial guess is not within the specified bounds
  result = minimize(


  Depth: 24.27eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 23.97eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 24.45eV, Offset: 0.4231mm, Power: 0.001675mW
  Depth: 24.19eV, Offset: 0.4231mm, Power: 0.001378mW
  Depth: 24.99eV, Offset: 0.4231mm, Power: 0.001675mW
  Depth: 24.75eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 24.27eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 24.27eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 24.27eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 24.27eV, Offset: 0.4231mm, Power: 0.001520mW
  Depth: 23.61eV, Offset: 0.4416mm, Power: 0.001675mW
  Depth: 25.11eV, Offset: 0.4047mm, Power: 0.001396mW
  Depth: 25.25eV, Offset: 0.4014mm, Power: 0.001372mW
  Depth: 25.02eV, Offset: 0.4191mm, Power: 0.001520mW
  Depth: 24.92eV, Offset: 0.4184mm, Power: 0.001714mW
  Depth: 24.97eV, Offset: 0.4176mm, Power: 0.001581mW
  Depth: 25.11eV, Offset: 0.4166mm, Power: 0.001592mW
  Depth: 25.23eV, Offset: 0.4154mm, Power: 0.001605mW
  Depth: 25.41eV, Offset: 0.

/tmp/ipykernel_155601/2884008902.py:252: RuntimeWarning: Maximum number of iterations has been exceeded.
  result = minimize(
